In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
print("PROJECT ROOT ==>", PROJECT_ROOT)

sys.path.append(str(PROJECT_ROOT))
from src.mesh_fun import (
    load_segmentation,load_pred_landmarks,visualize_pred_and_gt,visualize,load_mesh,color_mesh_by_groups,create_landmark_spheres,load_landmarks,
    load_gt_landmarks
)
from src.landmark_eval import( 
    compute_all_statistics,
    evaluate_single_scan_from_landmarks,
    quality_profile_for_scan,
select_examples_by_quantile, 
compute_profiles_for_scans)
from src.var_constants import CATEGORIES,TOOTH_GROUP_PALETTE,LANDMARK_PALETTE,TOOTH_TO_GROUP


PROJECT ROOT ==> C:\Users\aless\Tesi_Msc\multi_agent_evaluation_on_intraoral_3D_scans


In [7]:
#PROJECT_ROOT = Path(__file__).resolve().parent
DATA_ROOT = PROJECT_ROOT / "dataset"
SCANS = DATA_ROOT / "toothinstancenet_input"
GT_ROOT = SCANS
PRED_CSV = DATA_ROOT / "model_predictions" / "ynlab" / "predictions.csv"
SCREENSHOT_DIR = DATA_ROOT / "screenshot_scans" / "raw"

In [8]:
scan_name='AKBDPB4C_upper'

In [9]:
screenshot_mesh_path =SCREENSHOT_DIR / scan_name
mesh_path = SCANS / f"{scan_name}.obj"
gt_json = GT_ROOT / f"{scan_name}__kpt.json"
seg_path  =SCANS / f"{scan_name}_seg.json"
tooth_seg_labels = load_segmentation(seg_path)

#### funzioni di pertbuazione

In [ ]:
import random
import numpy as np

#Rimuove una percentuale di landmark
def apply_missing_landmark_perturbation(
    coords,
    classes,
    fraction,
    #specifica opzionalmente un sotto-insieme di classi da considerare. Es. Mesial
    target_landmark_classes=None,
    seed=42,
):
    rng = random.Random(seed)
    if len(coords) == 0:
        return coords.copy(), classes.copy()
    if target_landmark_classes is None:
        candidate_idx = list(range(len(coords)))
    else:
        candidate_idx = [i for i, cls in enumerate(classes) if cls in target_landmark_classes]
    if len(candidate_idx) == 0:
        return ( coords.copy(),classes.copy() )
    n_remove = max( 1, int(len(candidate_idx) * fraction))

    remove_idx = set(rng.sample(  candidate_idx, min(    n_remove,len(candidate_idx))) )
    keep_idx = [i for i in range(len(coords)) if i not in remove_idx]
    coords_new = coords[keep_idx]
    classes_new = [classes[i] for i in keep_idx]
    return (
        coords_new,
        classes_new,
    )

In [ ]:
coords_pred, classes_pred = load_pred_landmarks(PRED_CSV, scan_name)
coords_gt, classes_gt = load_gt_landmarks(gt_json)

noisy_coords, noisy_classes=apply_missing_landmark_perturbation(
    coords_pred,
    classes_pred,
    fraction=0.990,
    target_landmark_classes='Mesial',
    seed=42)
mesh = load_mesh(mesh_path)
mesh = color_mesh_by_groups(mesh, tooth_seg_labels, TOOTH_TO_GROUP, TOOTH_GROUP_PALETTE)
spheres_pred = create_landmark_spheres(coords_pred, classes_pred, LANDMARK_PALETTE)
spheres_gt = create_landmark_spheres(coords_gt, classes_gt, LANDMARK_PALETTE)

spheres_noisy = create_landmark_spheres(noisy_coords, noisy_classes, LANDMARK_PALETTE)

#visualize(mesh, spheres_pred, view="front")
#visualize(mesh, spheres_noisy, view="front")
scan_evaluation_=evaluate_single_scan_from_landmarks(scan_name=scan_name,
                                    coords_pred=coords_pred,
                                    classes_pred=classes_pred,
                                    coords_gt=coords_gt,
                                    classes_gt=classes_gt,categories=CATEGORIES)
scan_quality_profile=quality_profile_for_scan([scan_evaluation_],categories=CATEGORIES,scan_name=scan_name)


dict_keys(['scan', 'ap_per_class', 'ar_per_class', 'gt_count_per_class', 'pred_count_per_class', 'mAP', 'mAR'])

In [ ]:
#qui potrei aggiungere anche un max di campioni che hanno specifica qualità (es. quality=3 o quality=4) prima di passare
#ad un'altra configurazione
def build_synthetic_oracle_dataset(scan_names, SCANS, GT_ROOT, PRED_CSV,SCREENSHOT_ROOT,CATEGORIES,pertubation_configs):
    dataset = []
    for scan_name in scan_names: 
        mesh_path = SCANS / f"{scan_name}.obj"
        gt_json = GT_ROOT / f"{scan_name}__kpt.json"
        seg_path  =SCANS / f"{scan_name}_seg.json"
        tooth_seg_labels = load_segmentation(seg_path)

        mesh = load_mesh(mesh_path)
        mesh = color_mesh_by_groups(mesh, tooth_seg_labels, TOOTH_TO_GROUP, TOOTH_GROUP_PALETTE)    
        
        coords_pred, classes_pred = load_pred_landmarks(PRED_CSV, scan_name)
        coords_gt, classes_gt = load_gt_landmarks(gt_json)


In [ ]:
PERTURBATIONS = [
    {
        "type":
            "missing_landmarks",
        "params": {
            "fraction":0.10,
            "severity": "low",
            "target_landmark_classes": ["Mesial"],
        },
    }
]